In [1]:
import tensorflow as tf
from tensorflow.keras import datasets, layers, models
from tensorflow.keras.optimizers import Adam
import keras
from keras.models import Sequential, Model
from keras.layers import *
from keras.utils import Sequence
from keras.layers import Conv2D, MaxPooling2D
from qkeras import *

from keras.utils import Sequence
from keras.callbacks import CSVLogger
from keras.callbacks import EarlyStopping

import os
import random
from datetime import datetime
import time

import matplotlib.pyplot as plt

pi = 3.14159265359

maxval=1e9
minval=1e-9



2025-06-24 18:24:54.692251: E external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:9261] Unable to register cuDNN factory: Attempting to register factory for plugin cuDNN when one has already been registered
2025-06-24 18:24:54.692318: E external/local_xla/xla/stream_executor/cuda/cuda_fft.cc:607] Unable to register cuFFT factory: Attempting to register factory for plugin cuFFT when one has already been registered
2025-06-24 18:24:54.693199: E external/local_xla/xla/stream_executor/cuda/cuda_blas.cc:1515] Unable to register cuBLAS factory: Attempting to register factory for plugin cuBLAS when one has already been registered
2025-06-24 18:24:54.699851: I tensorflow/core/platform/cpu_feature_guard.cc:182] This TensorFlow binary is optimized to use available CPU instructions in performance-critical operations.
To enable the following instructions: AVX2 FMA, in other operations, rebuild TensorFlow with the appropriate compiler flags.
2025-06-24 18:24:56.646305: W tensorflow/compiler/tf2

In [2]:
# os.chdir('SmartPix/data_generator')
os.chdir('/home/das214/SmartPix/mlp_enc_dev')
!pwd

/home/das214/SmartPix/mlp_enc_dev


In [3]:
from DG.OptimizedDataGenerator_v2 import OptimizedDataGenerator
from losses.diag_loss_nll import custom_diag_loss
from models.mlp_encoder_model import CreateModel

In [4]:
dataset_base_dir = "/depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained"
tfrecords_base_dir = os.path.join(dataset_base_dir, "TFR_files", "2t")

dataset_train_dir = os.path.join(dataset_base_dir, "train")
dataset_test_dir = os.path.join(dataset_base_dir, "test")
tfrecords_dir_train = os.path.join(tfrecords_base_dir, "TFR_train")
tfrecords_dir_val   = os.path.join(tfrecords_base_dir, "TFR_val")

batch_size = 5000
val_batch_size = 5000
train_file_size = len(os.listdir(dataset_train_dir))
val_file_size = len(os.listdir(dataset_test_dir))

In [5]:
# start_time = time.time()
# validation_generator = OptimizedDataGenerator(
#     dataset_base_dir = dataset_test_dir,
#     file_type = "parquet",
#     data_format = "3D",
#     batch_size = val_batch_size,
#     # optimize_batch_size = True,
#     file_count = val_file_size,
#     to_standardize= True,
#     labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
#     input_shape = (2,16,16), # (20,13,21),
#     transpose = (0,2,3,1),
#     shuffle = False, 
#     files_from_end=True,

#     tfrecords_dir = tfrecords_dir_val,
#     use_time_stamps = [0,19],
#     max_workers = 2
# )

# print("--- Validation generator %s seconds ---" % (time.time() - start_time))

# # training generator
# start_time = time.time()
# training_generator = OptimizedDataGenerator(
#     dataset_base_dir = dataset_train_dir,
#     file_type = "parquet",
#     data_format = "3D",
#     batch_size = batch_size,
#     # optimize_batch_size = True,
#     file_count = train_file_size,
#     to_standardize= True,
#     labels_list = ['x-midplane','y-midplane','cotAlpha','cotBeta'],
#     input_shape = (2,16,16), # (20,13,21),
#     transpose = (0,2,3,1),
#     shuffle = False, # True 

#     tfrecords_dir = tfrecords_dir_train,
#     use_time_stamps = [0,19],
#     max_workers = 2
# )
# print("--- Training generator %s seconds ---" % (time.time() - start_time))

In [6]:
# Loading pre-generated TFRecords
validation_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir= tfrecords_dir_val,
    shuffle=True,
    seed=42,
    quantize=True,
)

training_generator = OptimizedDataGenerator(
    load_from_tfrecords_dir = tfrecords_dir_train,
    shuffle=True,
    seed=42,
    quantize=True,
)


Loading metadata from /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_val/metadata.json
Loading metadata from /depot/cms/users/das214/datasets/dataset_3sr/dataset_3sr_16x16_50x12P5_parquets/contained/TFR_files/2t/TFR_train/metadata.json


In [ ]:
diag_model=CreateModel(shape = (16,16,2), output = 8)
diag_model.compile(
    optimizer=tf.keras.optimizers.Nadam(learning_rate=1e-3),
    loss=custom_diag_loss,
    run_eagerly=True,
)

diag_model.summary()

2025-06-24 18:25:04.945667: I tensorflow/core/common_runtime/gpu/gpu_device.cc:1929] Created device /job:localhost/replica:0/task:0/device:GPU:0 with 38397 MB memory:  -> device: 0, name: NVIDIA A100-PCIE-40GB MIG 7g.40gb, pci bus id: 0000:21:00.0, compute capability: 8.0
2025-06-24 18:25:05.248180: I external/local_tsl/tsl/platform/default/subprocess.cc:304] Start cannot spawn child process: No such file or directory


Model: "smrtpxl_regression"
__________________________________________________________________________________________________
 Layer (type)                Output Shape                 Param #   Connected to                  
 input_pxls/ (InputLayer)    [(None, 16, 16, 2)]          0         []                            
                                                                                                  
 average_pooling2d (Average  (None, 16, 1, 2)             0         ['input_pxls/[0][0]']         
 Pooling2D)                                                                                       
                                                                                                  
 average_pooling2d_1 (Avera  (None, 1, 16, 2)             0         ['input_pxls/[0][0]']         
 gePooling2D)                                                                                     
                                                                                 

In [8]:
from datetime import datetime

fingerprint = '%08x' % random.randrange(16**8)
timestamp = datetime.now().strftime('%Y%m%d-%H%M%S')
os.makedirs("trained_models", exist_ok=True)
base_dir = f'./trained_models/model-{fingerprint}-checkpoints'
os.makedirs(base_dir, exist_ok=True)  
checkpoint_filepath = base_dir + '/weights.{epoch:02d}-t{loss:.2f}-v{val_loss:.2f}.hdf5'

In [9]:
print(fingerprint)

e4316b17


In [10]:
from tensorflow.keras.callbacks import CSVLogger, EarlyStopping, ModelCheckpoint, Callback

early_stopping_patience = 50

class CustomModelCheckpoint(ModelCheckpoint):
    def on_epoch_end(self, epoch, logs=None):
        super().on_epoch_end(epoch, logs)
        checkpoints = [f for f in os.listdir(base_dir) if f.startswith('weights')]
        if len(checkpoints) > 1:
            checkpoints.sort()
            for checkpoint in checkpoints[:-1]:
                os.remove(os.path.join(base_dir, checkpoint))

es = EarlyStopping(patience=early_stopping_patience, restore_best_weights=True)

mcp = CustomModelCheckpoint(
    filepath=checkpoint_filepath,
    save_weights_only=True,
    monitor='val_loss',
    save_best_only=True,
    save_freq='epoch',
    verbose=1
)

csv_logger = CSVLogger(f'{base_dir}/training_log.csv', append=True)

In [11]:
training_generator.__len__()

84

In [12]:
for i in range(training_generator.__len__()):

    X_batch, y_batch = training_generator[i]

    print("--- Running a single forward pass... ---")
    try:
        # Get the model's raw predictions
        predictions = diag_model(X_batch, training=True)

        # Use TensorFlow's built-in checker
        tf.debugging.check_numerics(predictions, "Model predictions contain NaN or Inf!")

        print("✅ SUCCESS: The model's raw output is numerically stable (no NaNs or Infs).")
        print("\nSample of predictions (first 5):")
        print(predictions.numpy()[:5])

    except Exception as e:
        print(f"❌ FAILURE: The NaN is being generated inside the model's forward pass.")
        print(f"Error: {e}")


--- Running a single forward pass... ---


2025-06-24 18:25:08.905632: I external/local_xla/xla/stream_executor/cuda/cuda_dnn.cc:454] Loaded cuDNN version 8907


✅ SUCCESS: The model's raw output is numerically stable (no NaNs or Infs).

Sample of predictions (first 5):
[[ 0.11688232 -0.2880249  -0.1508789  -0.07354736  0.12957764  0.09796143
   0.0256958  -0.17150879]
 [-0.14453125 -0.04748535  0.28863525 -0.10760498  0.1496582  -0.262146
   0.5358887   0.4821167 ]
 [-0.02770996  0.09265137  0.0255127  -0.02478027  0.17150879 -0.04956055
  -0.06616211  0.25683594]
 [ 0.05615234 -0.2744751   0.02032471 -0.24475098 -0.0135498  -0.03753662
   0.30474854 -0.07849121]
 [ 0.01611328 -0.1352539  -0.019104   -0.26153564 -0.03442383 -0.1463623
   0.5808716   0.24395752]]
--- Running a single forward pass... ---
✅ SUCCESS: The model's raw output is numerically stable (no NaNs or Infs).

Sample of predictions (first 5):
[[ 0.06298828  0.00128174 -0.25372314 -0.05682373  0.21014404  0.00604248
   0.16375732  0.32580566]
 [ 0.00738525 -0.12536621 -0.13867188 -0.0904541   0.1295166   0.00067139
   0.15478516  0.10925293]
 [-0.05438232  0.16308594  0.0550537

In [13]:
history = diag_model.fit(
        x=training_generator,
        validation_data=validation_generator,
        callbacks=[es, mcp, csv_logger],
        epochs=1000,
        shuffle=False,
        verbose=1
    )

Epoch 1/1000


2025-06-24 18:25:18.561716: I tensorflow/core/util/cuda_solvers.cc:179] Creating GpuSolver handles for stream 0x56064f6618d0
2025-06-24 18:25:19.460747: I external/local_xla/xla/service/service.cc:168] XLA service 0x56065b69d230 initialized for platform CUDA (this does not guarantee that XLA will be used). Devices:
2025-06-24 18:25:19.460797: I external/local_xla/xla/service/service.cc:176]   StreamExecutor device (0): NVIDIA A100-PCIE-40GB MIG 7g.40gb, Compute Capability 8.0
2025-06-24 18:25:19.471265: I tensorflow/compiler/mlir/tensorflow/utils/dump_mlir_util.cc:269] disabling MLIR crash reproducer, set env var `MLIR_CRASH_REPRODUCER_DIRECTORY` to enable.
I0000 00:00:1750782319.594646  469140 device_compiler.h:186] Compiled cluster using XLA!  This line is logged at most once for the lifetime of the process.


84/84 [==============================] - ETA: 0s - loss: 90.9585
Epoch 1: val_loss improved from inf to 90.17201, saving model to ./trained_models/model-e4316b17-checkpoints/weights.01-t90.96-v90.17.hdf5
84/84 [==============================] - 16s 150ms/step - loss: 90.9585 - val_loss: 90.1720
Epoch 2/1000
84/84 [==============================] - ETA: 0s - loss: 89.5331
Epoch 2: val_loss improved from 90.17201 to 88.92966, saving model to ./trained_models/model-e4316b17-checkpoints/weights.02-t89.53-v88.93.hdf5
84/84 [==============================] - 12s 148ms/step - loss: 89.5331 - val_loss: 88.9297
Epoch 3/1000
84/84 [==============================] - ETA: 0s - loss: 88.4530
Epoch 3: val_loss improved from 88.92966 to 88.00980, saving model to ./trained_models/model-e4316b17-checkpoints/weights.03-t88.45-v88.01.hdf5
84/84 [==============================] - 12s 146ms/step - loss: 88.4530 - val_loss: 88.0098
Epoch 4/1000
84/84 [==============================] - ETA: 0s - loss: 87.676

KeyboardInterrupt: 

In [ ]:
X_batch, y_batch = training_generator[0]

print("--- Running a single forward pass... ---")
try:
    # Get the model's raw predictions
    predictions = diag_model(X_batch, training=True)

    # Use TensorFlow's built-in checker
    tf.debugging.check_numerics(predictions, "Model predictions contain NaN or Inf!")

    print("✅ SUCCESS: The model's raw output is numerically stable (no NaNs or Infs).")
    print("\nSample of predictions (first 5):")
    print(predictions.numpy()[:5])

except Exception as e:
    print(f"❌ FAILURE: The NaN is being generated inside the model's forward pass.")
    print(f"Error: {e}")


--- Running a single forward pass... ---
❌ FAILURE: The NaN is being generated inside the model's forward pass.
Error: {{function_node __wrapped__CheckNumerics_device_/job:localhost/replica:0/task:0/device:GPU:0}} Model predictions contain NaN or Inf! : Tensor had NaN values [Op:CheckNumerics] name: 


2025-06-23 15:25:01.823355: E tensorflow/core/kernels/check_numerics_op.cc:293] abnormal_detected_host @0x7f44dfe00000 = {1, 0} Model predictions contain NaN or Inf!
